# KM Hallucination Pilot — Model Run Only (GPU)

**Version:** 1.0.1-gpu-models-only  
**Purpose:** run the four primary models over the frozen case manifest on a GPU and produce
material for **hallucination** error analysis.

This notebook contains **no** corpus profiling, sampling, case construction or dimension
induction. It consumes one input artefact produced by the main notebook:

```
km_pilot_outputs/pilot_case_manifest.json
```

## The distinction this notebook is built around

**A hallucination is not the same as a wrong answer.** The triage in Section 7 keeps them
in separate columns and never merges them:

| Class | Meaning | Example |
|---|---|---|
| `HALLUCINATION_CANDIDATE` | asserted content that is **not present in** or **contradicts** the supplied turns | cites turn `C7` when the thread only has `C1`-`C3`; describes what a linked page says; gives an occupation never stated |
| `NON_HALLUCINATION_ERROR_CANDIDATE` | wrong, incomplete or malformed, but every asserted element **does** exist in the context | picks `C2` when the gold is `C3`; lists 2 of 4 labels; emits prose instead of JSON |
| `MATCHES_PROVISIONAL_GOLD` | agrees with the machine-derived gold | — |
| `REQUIRES_HUMAN_LABEL` | conceptual task with no machine gold | all linguistic diagnostic families |

Picking the **wrong existing item** is a retrieval failure. Inventing an item that **does not
exist** is a grounding failure. Only the second is a hallucination, and only the second should
drive a hallucination benchmark dimension.

## Honesty rules retained from the main protocol

* Nothing here **assigns** a hallucination label. Section 7 emits **candidates** for human review.
* Results are **append-only** and resumable; an identical configuration is never re-run.
* Models run **strictly one at a time** and are unloaded before the next one loads.
* Outputs are exported **blinded** for annotation; the identity key is written separately.

In [6]:
# --- Setup: install the packages missing from a fresh GPU environment -------
import importlib.util as _importlib_util
import subprocess as _subprocess
import sys as _sys

_required = {
    'transformers': 'transformers>=4.51,<5',
    'accelerate': 'accelerate>=1.0',
    'huggingface_hub': 'huggingface_hub>=0.30',
    'safetensors': 'safetensors>=0.4',
    'sentencepiece': 'sentencepiece>=0.2',
    'pandas': 'pandas>=2.0',
    'openpyxl': 'openpyxl>=3.1',
}
_missing = [spec for module, spec in _required.items()
            if _importlib_util.find_spec(module) is None]
if _missing:
    print('Installing missing packages:', ', '.join(_missing), flush=True)
    _subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '--quiet', *_missing])
    print('Package installation complete.', flush=True)
else:
    print('Required packages are already installed.')

Required packages are already installed.


In [7]:
# --- Section 0: Hugging Face access ---------------------------------------
# On a normal GPU box leave USE_ARTIFACTORY_PROXY = False and export HF_TOKEN.
# Set it True only on the Rio Tinto corporate network, where huggingface.co is
# intercepted and the approved Artifactory HuggingFaceML proxy must be used.
import configparser as _cp
import os as _o
from urllib.parse import urlsplit as _usplit

USE_ARTIFACTORY_PROXY = False
ARTIFACTORY_HF_ENDPOINT = 'https://artifactory.riotinto.com/artifactory/api/huggingfaceml/huggingface'


def _artifactory_token():
    'Read the Artifactory credential from pip.ini. Returns the value only, never prints it.'
    for _p in (_o.path.join(_o.environ.get('USERPROFILE', ''), 'pip', 'pip.ini'),
               _o.path.join(_o.environ.get('APPDATA', ''), 'pip', 'pip.ini'),
               _o.path.expanduser('~/.pip/pip.conf'),
               _o.path.expanduser('~/.config/pip/pip.conf')):
        if _p and _o.path.isfile(_p):
            _c = _cp.ConfigParser()
            _c.read(_p)
            for _sec in ('global', 'install'):
                _url = _c.get(_sec, 'index-url', fallback='')
                if _url:
                    _parts = _usplit(_url)
                    if _parts.password:
                        return _parts.password
    return None


ACCESS = {'proxy_used': False, 'endpoint': 'https://huggingface.co', 'token_present': False}

if USE_ARTIFACTORY_PROXY:
    try:
        import truststore as _ts
        _ts.inject_into_ssl()
    except Exception:
        pass
    _tok = _artifactory_token()
    if _tok:
        _o.environ['HF_ENDPOINT'] = ARTIFACTORY_HF_ENDPOINT
        _o.environ['HF_TOKEN'] = _tok
        _o.environ['HUGGING_FACE_HUB_TOKEN'] = _tok
        ACCESS.update(proxy_used=True, endpoint=ARTIFACTORY_HF_ENDPOINT, token_present=True)
    del _tok
    _o.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '120')
    _o.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '600')
    _o.environ.setdefault('HF_HUB_DISABLE_XET', '1')
else:
    ACCESS['token_present'] = bool(_o.environ.get('HF_TOKEN')
                                   or _o.environ.get('HUGGING_FACE_HUB_TOKEN'))

_o.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

print('hf endpoint   :', ACCESS['endpoint'])
print('proxy used    :', ACCESS['proxy_used'])
print('token present :', ACCESS['token_present'], '(value never printed or stored)')
if not ACCESS['token_present']:
    print('\n[WARNING] No token found. google/gemma-3-4b-it is a gated repo and will fail.')
    print('          Export HF_TOKEN and accept the Gemma terms on the model page first.')

hf endpoint   : https://huggingface.co
proxy used    : False
token present : False (value never printed or stored)

[WARNING] No token found. google/gemma-3-4b-it is a gated repo and will fail.
          Export HF_TOKEN and accept the Gemma terms on the model page first.


In [8]:
import os

In [9]:
# --- Section 0b: secure token prompt for gated Gemma access -------------------
# Nothing is printed or written to the notebook. The token lives only in this kernel.
if not (os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')):
    import getpass

    _entered_token = getpass.getpass(
        'Hugging Face token required for Gemma (input hidden; press Enter to skip): '
    ).strip()
    if _entered_token:
        os.environ['HF_TOKEN'] = _entered_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = _entered_token
        ACCESS['token_present'] = True
        print('Hugging Face token loaded into this kernel (value hidden).')
    else:
        print('No token supplied. Public models may run, but Gemma will not.')
    del _entered_token
else:
    print('Hugging Face token is already available to this kernel.')

Hugging Face token required for Gemma (input hidden; press Enter to skip):  ········


Hugging Face token loaded into this kernel (value hidden).


In [10]:
# --- Section 1: frozen configuration --------------------------------------
import csv
import datetime as _dt
import gc
import hashlib
import json
import math
import os
import platform
import re
import sys
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

NOTEBOOK_NAME = 'km_hallucination_model_run_gpu.ipynb'
NOTEBOOK_VERSION = '1.0.1-gpu-models-only'
RANDOM_SEED_STRING = 'KM-PILOT-20260915'
RUN_STARTED_UTC = _dt.datetime.now(_dt.timezone.utc).isoformat()

# --- paths: set KM_PILOT_OUTPUT, or put this notebook beside km_pilot_outputs
_env_out = os.environ.get('KM_PILOT_OUTPUT')
if _env_out:
    OUTPUT_DIR = Path(_env_out).resolve()
else:
    _here = Path.cwd()
    _cands = [_here / 'km_pilot_outputs', _here, _here.parent / 'km_pilot_outputs',
              Path.home() / 'Downloads' / 'km_pilot_outputs']
    OUTPUT_DIR = next((p for p in _cands if (p / 'pilot_case_manifest.json').is_file()), None)
    if OUTPUT_DIR is None:
        _found = []
        for _root in (_here, _here.parent, Path('/workspace'), Path('/content')):
            if _root.is_dir():
                _found.extend(_root.rglob('pilot_case_manifest.json'))
        _found = sorted(set(p.resolve() for p in _found))
        OUTPUT_DIR = _found[0].parent if _found else _here / 'km_pilot_outputs'
    OUTPUT_DIR = OUTPUT_DIR.resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESTRICTED_DIR = OUTPUT_DIR / 'restricted'
RESTRICTED_DIR.mkdir(parents=True, exist_ok=True)
CASE_MANIFEST_PATH = OUTPUT_DIR / 'pilot_case_manifest.json'


def out(name):
    return OUTPUT_DIR / name


# --- what to run ----------------------------------------------------------
# Keep this True until all four models complete the five-case diagnostic.
# Then set it to False, restart the kernel, and run all cells for the study.
DIAGNOSTIC_RUN = True
LOAD_STRATEGY = 'BF16_GPU'      # BF16_GPU | FP16_GPU | QUANTIZED_4BIT_GPU
DEVICE_MAP = 'auto'
MAX_CASES = 5 if DIAGNOSTIC_RUN else None
MAX_INPUT_TOKENS = 4096
TOP_K_ALTERNATIVES = 5
MAX_UNCERTAINTY_STEPS = 256
# Full-vocabulary scores consume substantial time and memory. Capture them only
# after model execution is known to work on this GPU.
CAPTURE_LOGITS = not DIAGNOSTIC_RUN
RESUME_EXISTING_RUN = True      # never re-run an identical configuration
UNLOAD_MODEL_AFTER_RUN = True
MODEL_CACHE_DIR = os.environ.get('MODEL_CACHE_DIR') or os.environ.get('HF_HOME') or None

# Four primary research models. The 0.6B floor model is deliberately absent:
# it is a smoke test, never a research comparator.
MODELS = [
    {'model_key': 'qwen3_4b_instruct', 'model_id': 'Qwen/Qwen3-4B-Instruct-2507',
     'role': 'PRIMARY_1_NON_REASONING', 'thinking': False,
     'trust_remote_code': False, 'chat_template_kwargs': {}},
    {'model_key': 'qwen3_4b_thinking', 'model_id': 'Qwen/Qwen3-4B-Thinking-2507',
     'role': 'PRIMARY_2_REASONING', 'thinking': True,
     'trust_remote_code': False, 'chat_template_kwargs': {}},
    {'model_key': 'phi4_mini_instruct', 'model_id': 'microsoft/Phi-4-mini-instruct',
     'role': 'PRIMARY_3_INDEPENDENT_FAMILY', 'thinking': False,
     'trust_remote_code': False, 'chat_template_kwargs': {}},
    {'model_key': 'gemma3_4b_it', 'model_id': 'google/gemma-3-4b-it',
     'role': 'PRIMARY_4_INDEPENDENT_FAMILY', 'thinking': False,
     'trust_remote_code': False, 'chat_template_kwargs': {}},
]

# Deterministic greedy decoding. Sampling would make the run irreproducible.
DECODING_CONFIG = {'do_sample': False, 'num_beams': 1, 'greedy': True,
                   'temperature': 'unset', 'top_p': 'unset', 'top_k': 'unset'}

SYSTEM_INSTRUCTION = (
    'Use only the supplied conversation. Do not add external information. Preserve the '
    'language of requested spans. If the evidence is insufficient, return '
    'INSUFFICIENT_EVIDENCE. Do not infer identities behind PERSON placeholders.'
)
USER_TEMPLATE = 'CONVERSATION:\n{context}\n\nTASK:\n{instruction}'


def sha256_text(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def seeded_rank(*parts):
    payload = RANDOM_SEED_STRING + '|' + '|'.join(str(p) for p in parts)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def write_jsonl(path, rows, mode='w'):
    with open(path, mode, encoding='utf-8') as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + '\n')
    return Path(path)


def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        keys = []
        for r in rows:
            for k in r:
                if k not in keys:
                    keys.append(k)
        fieldnames = keys or ['empty']
    with open(path, 'w', encoding='utf-8', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction='ignore')
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, '') for k in fieldnames})
    return Path(path)


def write_json(path, obj):
    with open(path, 'w', encoding='utf-8') as fh:
        json.dump(obj, fh, ensure_ascii=False, indent=2, default=str)
    return Path(path)


DECODING_SIGNATURE = sha256_text(json.dumps(DECODING_CONFIG, sort_keys=True))[:16]
RUN_ID = 'RUN_%s_%s' % (_dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ'),
                        seeded_rank('run', RUN_STARTED_UTC)[:8])

print('notebook      :', NOTEBOOK_NAME, NOTEBOOK_VERSION)
print('run_id        :', RUN_ID)
print('output dir    :', OUTPUT_DIR)
print('case manifest :', CASE_MANIFEST_PATH, '| exists:', CASE_MANIFEST_PATH.is_file())
print('diagnostic run :', DIAGNOSTIC_RUN, '| max cases:', MAX_CASES)
print('capture logits :', CAPTURE_LOGITS)
print('load strategy :', LOAD_STRATEGY)
print('models        :', [m['model_key'] for m in MODELS])

notebook      : km_hallucination_model_run_gpu.ipynb 1.0.1-gpu-models-only
run_id        : RUN_20260917T151255Z_bb0035ac
output dir    : /workspace/mauritian_creole_nlp
case manifest : /workspace/mauritian_creole_nlp/pilot_case_manifest.json | exists: True
diagnostic run : True | max cases: 5
capture logits : False
load strategy : BF16_GPU
models        : ['qwen3_4b_instruct', 'qwen3_4b_thinking', 'phi4_mini_instruct', 'gemma3_4b_it']


In [11]:
# --- Section 2: GPU preflight ---------------------------------------------
PREFLIGHT = {'utc': _dt.datetime.now(_dt.timezone.utc).isoformat(),
             'blocking': [], 'warnings': []}


def _ver(name):
    try:
        return getattr(__import__(name), '__version__', 'unknown')
    except Exception as exc:
        return 'MISSING (%s)' % type(exc).__name__


PREFLIGHT['packages'] = {n: _ver(n) for n in
                         ('torch', 'transformers', 'accelerate', 'huggingface_hub',
                          'safetensors', 'sentencepiece', 'pandas', 'openpyxl',
                          'psutil', 'bitsandbytes')}
PREFLIGHT['python'] = sys.version.split()[0]
PREFLIGHT['platform'] = platform.platform()

try:
    import torch
    HAVE_TORCH = True
except Exception as exc:
    torch, HAVE_TORCH = None, False
    PREFLIGHT['blocking'].append('torch not importable: %s' % exc)

try:
    import transformers
    HAVE_TRANSFORMERS = True
except Exception as exc:
    transformers, HAVE_TRANSFORMERS = None, False
    PREFLIGHT['blocking'].append('transformers not importable: %s' % exc)

try:
    import pandas as pd
    HAVE_PANDAS = True
except Exception:
    pd, HAVE_PANDAS = None, False

gpu = {'cuda_available': False, 'device_count': 0, 'devices': [], 'bf16_supported': None}
if HAVE_TORCH:
    gpu['torch_version'] = torch.__version__
    gpu['cuda_available'] = bool(torch.cuda.is_available())
    gpu['cuda_version'] = getattr(torch.version, 'cuda', None)
    if gpu['cuda_available']:
        gpu['device_count'] = torch.cuda.device_count()
        for i in range(gpu['device_count']):
            p = torch.cuda.get_device_properties(i)
            free_b, total_b = torch.cuda.mem_get_info(i)
            gpu['devices'].append({'index': i, 'name': p.name,
                                   'total_vram_gb': round(p.total_memory / 1e9, 2),
                                   'free_vram_gb': round(free_b / 1e9, 2),
                                   'capability': '%d.%d' % (p.major, p.minor)})
        try:
            gpu['bf16_supported'] = bool(torch.cuda.is_bf16_supported())
        except Exception:
            gpu['bf16_supported'] = None
PREFLIGHT['gpu'] = gpu

MAX_FREE_VRAM_GB = max([d['free_vram_gb'] for d in gpu['devices']], default=0.0)
if not gpu['cuda_available']:
    PREFLIGHT['blocking'].append(
        'No CUDA device. This notebook is GPU-only by design; run it on the GPU box.')
elif MAX_FREE_VRAM_GB < 10:
    PREFLIGHT['warnings'].append(
        'Only %.1f GB free VRAM. A 4B checkpoint in bfloat16 needs roughly 9-10 GB plus KV '
        'cache; consider LOAD_STRATEGY = QUANTIZED_4BIT_GPU.' % MAX_FREE_VRAM_GB)
if LOAD_STRATEGY == 'BF16_GPU' and gpu.get('bf16_supported') is False:
    PREFLIGHT['warnings'].append('Device reports no bfloat16 support; use FP16_GPU instead.')
if LOAD_STRATEGY == 'QUANTIZED_4BIT_GPU' and PREFLIGHT['packages']['bitsandbytes'].startswith('MISSING'):
    PREFLIGHT['blocking'].append('QUANTIZED_4BIT_GPU requires bitsandbytes, which is missing.')

print('=' * 70)
print('GPU PREFLIGHT')
print('=' * 70)
print('python        :', PREFLIGHT['python'])
print('platform      :', PREFLIGHT['platform'])
print('cuda available:', gpu['cuda_available'], '| devices:', gpu['device_count'],
      '| cuda:', gpu.get('cuda_version'), '| bf16:', gpu.get('bf16_supported'))
for d in gpu['devices']:
    print('   GPU %d %-28s %6.1f GB total / %6.1f GB free (sm_%s)'
          % (d['index'], d['name'][:28], d['total_vram_gb'], d['free_vram_gb'], d['capability']))
print('packages      :')
for k, v in PREFLIGHT['packages'].items():
    print('   %-18s %s' % (k, v))
if PREFLIGHT['blocking']:
    print('\nBLOCKING:')
    for m in PREFLIGHT['blocking']:
        print('  !', m)
if PREFLIGHT['warnings']:
    print('\nWARNINGS:')
    for m in PREFLIGHT['warnings']:
        print('  -', m)

GPU PREFLIGHT
python        : 3.12.3
platform      : Linux-6.8.0-124-generic-x86_64-with-glibc2.39
cuda available: True | devices: 1 | cuda: 12.8 | bf16: True
   GPU 0 NVIDIA GeForce RTX 4090        25.2 GB total /   24.8 GB free (sm_8.9)
packages      :
   torch              2.8.0+cu128
   transformers       4.57.6
   accelerate         1.15.0
   huggingface_hub    0.36.2
   safetensors        0.8.0
   sentencepiece      0.2.2
   pandas             3.0.5
   openpyxl           3.1.5
   psutil             7.1.0
   bitsandbytes       MISSING (ModuleNotFoundError)


In [12]:
# --- Section 3: load the frozen case manifest -----------------------------
# The manifest is the contract. Every model receives exactly these cases, in the
# same order, with the same instructions, evidence and token budgets.
if not CASE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        'pilot_case_manifest.json was not found. Upload it to %s, then rerun Cells 3-6. '
        'Alternatively set KM_PILOT_OUTPUT to the folder containing it.' % CASE_MANIFEST_PATH.parent)

with open(CASE_MANIFEST_PATH, encoding='utf-8') as fh:
    CASE_MANIFEST = json.load(fh)

CASES = list(CASE_MANIFEST.get('cases') or [])
PROMPT_VERSION = CASE_MANIFEST.get('prompt_version', 'UNKNOWN')
DATASET_SHA256 = CASE_MANIFEST.get('dataset_sha256', 'UNKNOWN')
MANIFEST_SHA256 = CASE_MANIFEST.get('manifest_sha256', 'UNKNOWN')

if CASE_MANIFEST.get('content_status') != 'OK':
    print('[WARNING] manifest content_status is %r, not OK.'
          % CASE_MANIFEST.get('content_status'))

# deterministic run order, identical for every model
CASES.sort(key=lambda c: seeded_rank('runorder', c['case_id']))
if MAX_CASES:
    CASES = CASES[:MAX_CASES]

CASE_BY_ID = {c['case_id']: c for c in CASES}

# Cases whose gold answer is conceptual carry no machine gold at all; they are the
# families where hallucination is actually expected to show up.
CONCEPTUAL_CASES = [c for c in CASES if c.get('gold_type') != 'MACHINE_DERIVED_VERIFIABLE']
ABSTENTION_CASES = [c for c in CASES if c.get('answerability_label') == 'INSUFFICIENT']

print('manifest sha256   :', MANIFEST_SHA256[:16])
print('dataset sha256    :', DATASET_SHA256[:16])
print('prompt version    :', PROMPT_VERSION)
print('cases to run      :', len(CASES), 'of', CASE_MANIFEST.get('case_count'))
print('threads covered   :', len({c['thread_id'] for c in CASES}))
print('posts covered     :', len({c.get('post_id') for c in CASES}))
print('\nby task class     :', dict(Counter(c['task_class'] for c in CASES)))
print('no machine gold   :', len(CONCEPTUAL_CASES), '(human judgement required)')
print('abstention-expected:', len(ABSTENTION_CASES), '(answering at all may be a hallucination)')
print('\nby task family:')
for fam, n in sorted(Counter(c['task_family'] for c in CASES).items()):
    print('   %-34s %4d' % (fam, n))
print('\ntotal generations planned: %d cases x %d models = %d'
      % (len(CASES), len(MODELS), len(CASES) * len(MODELS)))

manifest sha256   : 66265567a4fcd8b2
dataset sha256    : f9c0e53d6b7931a0
prompt version    : km-pilot-prompt-v1
cases to run      : 5 of 240
threads covered   : 5
posts covered     : 5

by task class     : {'LINGUISTIC_DIAGNOSTIC': 3, 'PIPELINE_CONTROL': 2}
no machine gold   : 3 (human judgement required)
abstention-expected: 0 (answering at all may be a hallucination)

by task family:
   answerability_abstention              1
   attribution                           1
   explicit_claim_extraction             1
   reference_resolution                  2

total generations planned: 5 cases x 4 models = 20


In [13]:
# --- Section 4: live model verification -----------------------------------
# Records what the Hub actually says. A model that cannot be verified is not run.
VERIFICATION_ROWS = []


def verify(row):
    rec = {'model_key': row['model_key'], 'model_id': row['model_id'], 'role': row['role'],
           'repo_exists': False, 'gated': 'UNKNOWN', 'licence': None, 'revision': None,
           'parameter_count': None, 'architecture': None, 'chat_template_present': None,
           'tokenizer_is_fast': None, 'status': 'UNVERIFIED', 'error': ''}
    token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
    try:
        from huggingface_hub import HfApi
        info = HfApi().model_info(row['model_id'], token=token, files_metadata=False)
        rec['repo_exists'] = True
        rec['revision'] = info.sha
        rec['gated'] = getattr(info, 'gated', 'UNKNOWN')
        card = getattr(info, 'card_data', None) or {}
        rec['licence'] = card.get('license') if hasattr(card, 'get') else None
        cfg = getattr(info, 'config', None) or {}
        archs = cfg.get('architectures') if isinstance(cfg, dict) else None
        rec['architecture'] = (archs or [None])[0]
        st = getattr(info, 'safetensors', None)
        rec['parameter_count'] = getattr(st, 'total', None) if st else None
        rec['status'] = 'VERIFIED_HUB_LIVE'
    except Exception as exc:
        rec['error'] = '%s: %s' % (type(exc).__name__, str(exc)[:200])
        VERIFICATION_ROWS.append(rec)
        return rec
    try:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(row['model_id'], token=token,
                                            cache_dir=MODEL_CACHE_DIR)
        rec['chat_template_present'] = bool(getattr(tok, 'chat_template', None))
        rec['tokenizer_is_fast'] = bool(getattr(tok, 'is_fast', False))
    except Exception as exc:
        rec['error'] = (rec['error'] + ' | tokenizer: %s' % str(exc)[:160]).strip(' |')
    VERIFICATION_ROWS.append(rec)
    return rec


print('%-34s %-20s %-8s %14s  %s' % ('model_id', 'status', 'gated', 'params', 'error'))
for _m in MODELS:
    print('verifying/downloading tokenizer:', _m['model_id'], flush=True)
    _r = verify(_m)
    _m['verification_status'] = _r['status']
    _m['model_revision'] = _r['revision']
    _m['gated'] = _r['gated']
    print('%-34s %-20s %-8s %14s  %s'
          % (_r['model_id'][:34], _r['status'], _r['gated'],
             _r['parameter_count'] or '-', _r['error'][:40]))

write_csv(out('gpu_model_verification.csv'), VERIFICATION_ROWS)

RUNNABLE = [m for m in MODELS if m['verification_status'] == 'VERIFIED_HUB_LIVE']
print('\nrunnable models: %d / %d' % (len(RUNNABLE), len(MODELS)))
if len(RUNNABLE) < len(MODELS):
    print('[OBSERVED] Not all four models are available. The run will proceed with those')
    print('           that verified, and the missing ones are recorded as not run.')
    print('           A four-way paired comparison is NOT possible until all four succeed.')

model_id                           status               gated            params  error
verifying/downloading tokenizer: Qwen/Qwen3-4B-Instruct-2507
Qwen/Qwen3-4B-Instruct-2507        VERIFIED_HUB_LIVE    False        4022468096  tokenizer: Fast download using 'hf_trans
verifying/downloading tokenizer: Qwen/Qwen3-4B-Thinking-2507
Qwen/Qwen3-4B-Thinking-2507        VERIFIED_HUB_LIVE    False        4022468096  tokenizer: Fast download using 'hf_trans
verifying/downloading tokenizer: microsoft/Phi-4-mini-instruct
microsoft/Phi-4-mini-instruct      VERIFIED_HUB_LIVE    False        3836021760  tokenizer: Fast download using 'hf_trans
verifying/downloading tokenizer: google/gemma-3-4b-it
google/gemma-3-4b-it               VERIFIED_HUB_LIVE    manual       4300079472  tokenizer: Fast download using 'hf_trans

runnable models: 4 / 4


In [14]:
# --- Section 5: loading, generation, uncertainty capture ------------------
_LOADED = {}
LOAD_ERRORS, MEMORY_LOG = [], []


def memory_snapshot(tag, model_key=''):
    snap = {'tag': tag, 'model_key': model_key,
            'utc': _dt.datetime.now(_dt.timezone.utc).isoformat(), 'gpu': []}
    if HAVE_TORCH and torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            free_b, total_b = torch.cuda.mem_get_info(i)
            snap['gpu'].append({'index': i, 'free_gb': round(free_b / 1e9, 2),
                                'allocated_gb': round(torch.cuda.memory_allocated(i) / 1e9, 2),
                                'reserved_gb': round(torch.cuda.memory_reserved(i) / 1e9, 2)})
    MEMORY_LOG.append(snap)
    return snap


def build_load_kwargs(row):
    q = {'strategy': LOAD_STRATEGY, 'quantization_method': None, 'storage_dtype': None,
         'compute_dtype': None, 'device_map': DEVICE_MAP, 'warnings': []}
    kw = {'cache_dir': MODEL_CACHE_DIR, 'device_map': DEVICE_MAP}
    if row.get('trust_remote_code'):
        kw['trust_remote_code'] = True
    if LOAD_STRATEGY == 'BF16_GPU':
        kw['dtype'] = torch.bfloat16
        q.update(storage_dtype='bfloat16', compute_dtype='bfloat16')
    elif LOAD_STRATEGY == 'FP16_GPU':
        kw['dtype'] = torch.float16
        q.update(storage_dtype='float16', compute_dtype='float16')
    elif LOAD_STRATEGY == 'QUANTIZED_4BIT_GPU':
        from transformers import BitsAndBytesConfig
        kw['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
        q.update(quantization_method='bitsandbytes_nf4', storage_dtype='nf4',
                 compute_dtype='bfloat16')
        q['warnings'].append('QUANTIZED RESULTS ARE A SEPARATE EXPERIMENTAL CONDITION: '
                             'do not pool them with full-precision results.')
    else:
        raise RuntimeError('unknown LOAD_STRATEGY: %s' % LOAD_STRATEGY)
    return kw, q


def load_model(row):
    key = row['model_key']
    if key in _LOADED:
        return _LOADED[key]
    from transformers import AutoModelForCausalLM, AutoTokenizer
    memory_snapshot('before_load', key)
    print('   downloading/loading tokenizer...', flush=True)
    token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
    tok_kwargs = {'cache_dir': MODEL_CACHE_DIR}
    if token:
        tok_kwargs['token'] = token
    if row.get('trust_remote_code'):
        tok_kwargs['trust_remote_code'] = True
    tok = AutoTokenizer.from_pretrained(row['model_id'], **tok_kwargs)
    kw, quant = build_load_kwargs(row)
    if token:
        kw['token'] = token
    print('   downloading/loading model weights (first run can take several minutes)...',
          flush=True)
    try:
        model = AutoModelForCausalLM.from_pretrained(row['model_id'], **kw)
    except TypeError:
        # transformers < 4.56 names this torch_dtype. Rename it rather than dropping it:
        # dropping it would silently load in float32 and need ~16 GB of VRAM for a 4B model.
        _dt_ = kw.pop('dtype', None)
        if _dt_ is not None:
            kw['torch_dtype'] = _dt_
        model = AutoModelForCausalLM.from_pretrained(row['model_id'], **kw)
    model.eval()
    cfg = getattr(model, 'config', None)
    bundle = {'row': row, 'model': model, 'tokenizer': tok, 'quantization': quant,
              'model_revision': getattr(cfg, '_commit_hash', None) or row.get('model_revision')
              or 'UNRESOLVED',
              'tokenizer_revision': getattr(tok, '_commit_hash', None) or 'UNRESOLVED',
              'device': str(next(model.parameters()).device),
              'tokenizer_is_fast': bool(getattr(tok, 'is_fast', False))}
    memory_snapshot('after_load', key)
    _LOADED[key] = bundle
    return bundle


def unload_model(key):
    b = _LOADED.pop(key, None)
    if b is None:
        return
    b['model'] = None
    b['tokenizer'] = None
    del b
    gc.collect()
    if HAVE_TORCH and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    memory_snapshot('after_unload', key)


def safe_load(row):
    try:
        return load_model(row)
    except Exception as exc:
        LOAD_ERRORS.append({'model_id': row['model_id'], 'model_key': row['model_key'],
                            'error': '%s: %s' % (type(exc).__name__, exc)})
        print('   LOAD FAILED: %s -> %s' % (row['model_id'], str(exc)[:200]))
        return None


# --- prompt rendering -----------------------------------------------------
def build_user_message(case):
    return USER_TEMPLATE.format(context=case['input_context'],
                                instruction=case['instruction'])


def render_prompt(bundle, case):
    tok = bundle['tokenizer']
    messages = [{'role': 'system', 'content': SYSTEM_INSTRUCTION},
                {'role': 'user', 'content': build_user_message(case)}]
    tmpl_kwargs = dict(bundle['row'].get('chat_template_kwargs') or {})
    if getattr(tok, 'chat_template', None):
        try:
            return tok.apply_chat_template(messages, tokenize=False,
                                           add_generation_prompt=True,
                                           **tmpl_kwargs), 'chat_template'
        except TypeError:
            return tok.apply_chat_template(messages, tokenize=False,
                                           add_generation_prompt=True), 'chat_template_no_kwargs'
        except Exception:
            pass
        # Gemma and some others reject a system role: merge it into the user turn.
        merged = [{'role': 'user',
                   'content': SYSTEM_INSTRUCTION + '\n\n' + messages[1]['content']}]
        try:
            return tok.apply_chat_template(merged, tokenize=False,
                                           add_generation_prompt=True), 'chat_template_merged_system'
        except Exception:
            pass
    return ('SYSTEM: %s\n\n%s\n\nASSISTANT:'
            % (SYSTEM_INSTRUCTION, build_user_message(case))), 'plain_fallback'


# --- output parsing -------------------------------------------------------
def parse_json_output(text):
    if text is None:
        return None, 'no_output'
    s = text.strip()
    try:
        return json.loads(s), 'direct'
    except Exception:
        pass
    fence = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', s, re.DOTALL)
    if fence:
        try:
            return json.loads(fence.group(1)), 'fenced'
        except Exception:
            pass
    brace = re.search(r'\{.*\}', s, re.DOTALL)
    if brace:
        try:
            return json.loads(brace.group(0)), 'braced'
        except Exception:
            pass
    return None, 'unparseable'


def strip_thinking(text):
    if text is None:
        return None, False
    cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    return cleaned, cleaned != (text or '').strip()


def exact_run_signature(case, row, bundle, prompt_hash, method):
    payload = {'case_id': case['case_id'], 'model_id': row['model_id'],
               'model_revision': str(bundle['model_revision']),
               'tokenizer_revision': str(bundle['tokenizer_revision']),
               'prompt_version': PROMPT_VERSION, 'rendered_prompt_sha256': prompt_hash,
               'task_family': case['task_family'],
               'max_input_tokens': MAX_INPUT_TOKENS,
               'max_new_tokens': case.get('max_new_tokens', 128),
               'decoding': DECODING_CONFIG, 'load_strategy': LOAD_STRATEGY,
               'quantization': bundle['quantization'], 'chat_template_method': method}
    return sha256_text(json.dumps(payload, sort_keys=True, default=str))


NUMBER_RE = re.compile(r'\d+')


def generate_one(bundle, case):
    row, tok, model = bundle['row'], bundle['tokenizer'], bundle['model']
    prompt, method = render_prompt(bundle, case)
    max_new = case.get('max_new_tokens', 128)
    enc = tok(prompt, return_tensors='pt', truncation=True, max_length=MAX_INPUT_TOKENS)
    input_ids = enc['input_ids']
    attention_mask = enc.get('attention_mask')
    if attention_mask is None:
        attention_mask = torch.ones_like(input_ids)
    in_len = int(input_ids.shape[-1])
    input_truncated = bool(len(tok(prompt)['input_ids']) > in_len)
    dev = bundle['device']
    gen_inputs = {'input_ids': input_ids.to(dev), 'attention_mask': attention_mask.to(dev)}
    gen_kwargs = {'max_new_tokens': max_new, 'do_sample': False, 'num_beams': 1,
                  'return_dict_in_generate': True, 'output_scores': bool(CAPTURE_LOGITS),
                  'pad_token_id': getattr(tok, 'pad_token_id', None)
                  or getattr(tok, 'eos_token_id', None)}
    err, gen = None, None
    t0 = _dt.datetime.now(_dt.timezone.utc)
    try:
        with torch.no_grad():
            gen = model.generate(**gen_inputs, **gen_kwargs)
    except Exception as exc:
        err = '%s: %s' % (type(exc).__name__, exc)
        if 'out of memory' in str(exc).lower():
            err = 'OOM: ' + err
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
    t1 = _dt.datetime.now(_dt.timezone.utc)

    unc_rows, raw, token_ids, out_len = [], None, [], 0
    mean_lp = min_lp = mean_ent = max_ent = None
    scores_available = False
    if gen is not None:
        seq = gen.sequences[0]
        token_ids = [int(x) for x in seq[in_len:].tolist()]
        out_len = len(token_ids)
        raw = tok.decode(token_ids, skip_special_tokens=True)
        scores = getattr(gen, 'scores', None)
        scores_available = bool(scores)
        if CAPTURE_LOGITS and scores:
            lps, ents = [], []
            for step, score in enumerate(scores[:MAX_UNCERTAINTY_STEPS]):
                if step >= len(token_ids):
                    break
                logprobs = torch.log_softmax(score[0].float(), dim=-1)
                probs = logprobs.exp()
                ent = float(-(probs * logprobs).nan_to_num(0.0).sum())
                tid = token_ids[step]
                lp = float(logprobs[tid])
                topk = torch.topk(logprobs, k=min(TOP_K_ALTERNATIVES, logprobs.shape[-1]))
                piece = tok.decode([tid])
                unc_rows.append({
                    'run_id': RUN_ID, 'case_id': case['case_id'],
                    'model_id': row['model_id'], 'model_key': row['model_key'],
                    'task_family': case['task_family'],
                    'step': step, 'token_id': tid, 'token_text': piece,
                    'selected_token_logprob': lp,
                    'selected_token_prob': float(math.exp(lp)),
                    'vocab_entropy_nats': ent,
                    'top_k_tokens': json.dumps([tok.decode([int(i)])
                                                for i in topk.indices.tolist()],
                                               ensure_ascii=False),
                    'top_k_logprobs': json.dumps([round(float(v), 6)
                                                  for v in topk.values.tolist()]),
                    'is_numeric_token': bool(NUMBER_RE.fullmatch(piece.strip() or 'x')),
                    'is_placeholder_fragment': 'PERSON' in piece})
                lps.append(lp)
                ents.append(ent)
            if lps:
                mean_lp, min_lp = sum(lps) / len(lps), min(lps)
                mean_ent, max_ent = sum(ents) / len(ents), max(ents)

    visible, had_think = strip_thinking(raw)
    parsed, parse_method = parse_json_output(visible)
    eos = getattr(tok, 'eos_token_id', None)
    truncated = bool(out_len >= max_new and (not token_ids or token_ids[-1] != eos))
    prompt_hash = sha256_text(prompt)

    result = {
        'run_id': RUN_ID, 'case_id': case['case_id'], 'thread_id': case['thread_id'],
        'post_id': case.get('post_id'), 'stratum': case.get('stratum'),
        'task_family': case['task_family'], 'task_class': case['task_class'],
        'task_variant': case.get('task_variant'),
        'answerability_label': case.get('answerability_label'),
        'gold_type': case.get('gold_type'),
        'model_id': row['model_id'], 'model_key': row['model_key'], 'role': row['role'],
        'model_revision': bundle['model_revision'],
        'tokenizer_revision': bundle['tokenizer_revision'],
        'load_strategy': LOAD_STRATEGY, 'quantization': bundle['quantization'],
        'prompt_version': PROMPT_VERSION, 'prompt_rendering_method': method,
        'rendered_prompt_sha256': prompt_hash,
        'input_token_ids': [int(x) for x in input_ids[0].tolist()],
        'attention_mask': [int(x) for x in attention_mask[0].tolist()],
        'input_token_count': in_len, 'input_truncated': input_truncated,
        'raw_output': raw, 'visible_output_after_thinking_strip': visible,
        'contained_thinking_block': had_think,
        'parsed_output': parsed, 'parse_method': parse_method,
        'generated_token_ids': token_ids, 'output_token_count': out_len,
        'max_new_tokens': max_new, 'decoding': dict(DECODING_CONFIG),
        'decoding_signature': DECODING_SIGNATURE,
        'generation_scores_available': scores_available,
        'start_utc': t0.isoformat(), 'end_utc': t1.isoformat(),
        'duration_seconds': round((t1 - t0).total_seconds(), 3),
        'stop_reason': 'error' if err else ('length' if truncated else 'eos_or_natural'),
        'truncated': truncated, 'runtime_error': err, 'device': bundle['device'],
        'mean_token_logprob': mean_lp, 'min_token_logprob': min_lp,
        'mean_vocab_entropy': mean_ent, 'max_vocab_entropy': max_ent,
    }
    result['exact_run_signature'] = exact_run_signature(case, row, bundle, prompt_hash, method)
    return result, unc_rows


print('Loader and generator ready. Greedy decoding; exact input_ids stored; append-only.')

Loader and generator ready. Greedy decoding; exact input_ids stored; append-only.


In [15]:
# --- Section 5b: Gemma 3 architecture-specific loader -------------------------
# Gemma 3 4B declares Gemma3ForConditionalGeneration, not a causal-LM class.
# Keep the common loader for Qwen/Phi and use the declared class for Gemma.
_load_causal_model = load_model


def load_model(row):
    if row['model_id'] != 'google/gemma-3-4b-it':
        return _load_causal_model(row)

    key = row['model_key']
    if key in _LOADED:
        return _LOADED[key]

    from transformers import AutoTokenizer, Gemma3ForConditionalGeneration

    memory_snapshot('before_load', key)
    print('   downloading/loading Gemma tokenizer...', flush=True)
    token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
    tok_kwargs = {'cache_dir': MODEL_CACHE_DIR}
    if token:
        tok_kwargs['token'] = token
    tok = AutoTokenizer.from_pretrained(row['model_id'], **tok_kwargs)

    kw, quant = build_load_kwargs(row)
    if token:
        kw['token'] = token
    print('   downloading/loading Gemma3ForConditionalGeneration weights...', flush=True)
    try:
        model = Gemma3ForConditionalGeneration.from_pretrained(row['model_id'], **kw)
    except TypeError:
        dtype = kw.pop('dtype', None)
        if dtype is not None:
            kw['torch_dtype'] = dtype
        model = Gemma3ForConditionalGeneration.from_pretrained(row['model_id'], **kw)

    model.eval()
    cfg = getattr(model, 'config', None)
    bundle = {
        'row': row,
        'model': model,
        'tokenizer': tok,
        'quantization': quant,
        'model_revision': (
            getattr(cfg, '_commit_hash', None)
            or row.get('model_revision')
            or 'UNRESOLVED'
        ),
        'tokenizer_revision': getattr(tok, '_commit_hash', None) or 'UNRESOLVED',
        'device': str(next(model.parameters()).device),
        'tokenizer_is_fast': bool(getattr(tok, 'is_fast', False)),
    }
    memory_snapshot('after_load', key)
    _LOADED[key] = bundle
    return bundle


print('Gemma 3 loader ready: Gemma3ForConditionalGeneration.')

Gemma 3 loader ready: Gemma3ForConditionalGeneration.


In [16]:
# --- Section 6: sequential execution --------------------------------------
# One model at a time: load -> run every case -> write immediately -> unload.
# Stopping the notebook mid-run loses nothing; re-running resumes.
RESULTS_PATH = out('pilot_inference_results.jsonl')
UNCERTAINTY_PATH = out('pilot_token_uncertainty.jsonl')


def load_existing_signatures():
    sigs = set()
    if RESULTS_PATH.exists():
        with open(RESULTS_PATH, encoding='utf-8') as fh:
            for line in fh:
                try:
                    r = json.loads(line)
                except Exception:
                    continue
                if r.get('runtime_error') or not r.get('exact_run_signature'):
                    continue
                sigs.add(r['exact_run_signature'])
    return sigs


EXISTING_SIGNATURES = load_existing_signatures()
print('already-completed configurations on disk:', len(EXISTING_SIGNATURES))

INFERENCE_RESULTS, UNCERTAINTY_ROWS = [], []
COMPLETENESS_ROWS, RUN_ERRORS = [], []

if not RUNNABLE:
    print('\nNo verified models. Nothing to run.')
elif PREFLIGHT['blocking']:
    print('\nPreflight is blocking; resolve the items above before running.')
else:
    for row in RUNNABLE:                                  # STRICTLY SEQUENTIAL
        mk = row['model_key']
        print('\n' + '=' * 70)
        print('%s  (%s)' % (row['model_id'], row['role']))
        print('=' * 70)
        bundle = safe_load(row)
        if bundle is None:
            for case in CASES:
                COMPLETENESS_ROWS.append({
                    'case_id': case['case_id'], 'model_id': row['model_id'],
                    'completed': False, 'failed': True, 'skipped': False,
                    'reason': 'MODEL_LOAD_FAILED'})
            continue
        print('   loaded on %s | revision %s' % (bundle['device'],
                                                 str(bundle['model_revision'])[:12]), flush=True)
        memory_snapshot('before_inference', mk)
        _t_start = _dt.datetime.now(_dt.timezone.utc)
        for idx, case in enumerate(CASES, 1):
            print('   generating case %d/%d: %s'
                  % (idx, len(CASES), case['case_id']), flush=True)
            try:
                res, unc = generate_one(bundle, case)
                sig = res['exact_run_signature']
                if RESUME_EXISTING_RUN and sig in EXISTING_SIGNATURES:
                    COMPLETENESS_ROWS.append({
                        'case_id': case['case_id'], 'model_id': row['model_id'],
                        'completed': True, 'failed': False, 'skipped': True,
                        'reason': 'ALREADY_COMPLETED_EXACT_CONFIGURATION'})
                    continue
                write_jsonl(RESULTS_PATH, [res], mode='a')
                if unc:
                    write_jsonl(UNCERTAINTY_PATH, unc, mode='a')
                INFERENCE_RESULTS.append(res)
                UNCERTAINTY_ROWS.extend(unc)
                EXISTING_SIGNATURES.add(sig)
                COMPLETENESS_ROWS.append({
                    'case_id': case['case_id'], 'model_id': row['model_id'],
                    'completed': not bool(res['runtime_error']),
                    'failed': bool(res['runtime_error']), 'skipped': False,
                    'reason': res['runtime_error'] or ''})
            except Exception as exc:
                RUN_ERRORS.append({'model_id': row['model_id'], 'case_id': case['case_id'],
                                   'error': '%s: %s' % (type(exc).__name__, exc)})
                COMPLETENESS_ROWS.append({
                    'case_id': case['case_id'], 'model_id': row['model_id'],
                    'completed': False, 'failed': True, 'skipped': False,
                    'reason': '%s: %s' % (type(exc).__name__, str(exc)[:120])})
            if DIAGNOSTIC_RUN or idx % 20 == 0 or idx == len(CASES):
                _el = (_dt.datetime.now(_dt.timezone.utc) - _t_start).total_seconds()
                _rate = _el / max(idx, 1)
                print('   %4d/%d cases | %.1f s/case | eta %.1f min'
                      % (idx, len(CASES), _rate, _rate * (len(CASES) - idx) / 60.0))
        memory_snapshot('after_inference', mk)
        if UNLOAD_MODEL_AFTER_RUN:
            unload_model(mk)
            print('   unloaded', row['model_id'])

for _p in (RESULTS_PATH, UNCERTAINTY_PATH):
    if not _p.exists():
        write_jsonl(_p, [])
write_csv(out('gpu_run_completeness.csv'), COMPLETENESS_ROWS)
write_json(out('gpu_memory_log.json'), MEMORY_LOG)

print('\n' + '=' * 70)
print('generations this run :', len(INFERENCE_RESULTS))
print('uncertainty rows     :', len(UNCERTAINTY_ROWS))
print('runtime errors       :', len(RUN_ERRORS), '| load errors:', len(LOAD_ERRORS))
print('results appended to  :', RESULTS_PATH)

already-completed configurations on disk: 0

Qwen/Qwen3-4B-Instruct-2507  (PRIMARY_1_NON_REASONING)
   downloading/loading tokenizer...
   LOAD FAILED: Qwen/Qwen3-4B-Instruct-2507 -> Fast download using 'hf_transfer' is enabled (HF_HUB_ENABLE_HF_TRANSFER=1) but 'hf_transfer' package is not available in your environment. Try `pip install hf_transfer`.

Qwen/Qwen3-4B-Thinking-2507  (PRIMARY_2_REASONING)
   downloading/loading tokenizer...
   LOAD FAILED: Qwen/Qwen3-4B-Thinking-2507 -> Fast download using 'hf_transfer' is enabled (HF_HUB_ENABLE_HF_TRANSFER=1) but 'hf_transfer' package is not available in your environment. Try `pip install hf_transfer`.

microsoft/Phi-4-mini-instruct  (PRIMARY_3_INDEPENDENT_FAMILY)
   downloading/loading tokenizer...
   LOAD FAILED: microsoft/Phi-4-mini-instruct -> Fast download using 'hf_transfer' is enabled (HF_HUB_ENABLE_HF_TRANSFER=1) but 'hf_transfer' package is not available in your environment. Try `pip install hf_transfer`.

google/gemma-3-4b-it  (

In [17]:
# --- Section 7: hallucination triage --------------------------------------
# THE CORE OF THIS NOTEBOOK.
#
# A hallucination is asserted content that is NOT PRESENT IN, or CONTRADICTS, the
# supplied turns. A wrong answer is content that IS present but is the wrong choice,
# incomplete, or badly formatted. These are separated below and never merged.
#
# Everything emitted here is a CANDIDATE for human review. No hallucination label is
# assigned automatically: string absence is evidence of ungroundedness, not proof of it,
# and a human must confirm every case.

LABEL_RE = re.compile(r'\b[CR]\d{1,3}\b')
PLACEHOLDER_RE = re.compile(r'PERSON_\d{6}')
SPEAKER_RE = re.compile(r'\bS\d{5,9}\b')
URL_RE = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
PROPER_RE = re.compile(r'\b[A-Z][a-z]{3,}\b')

ABSTENTION_MARKERS = ('INSUFFICIENT_EVIDENCE', 'LINK_CONTENT_NOT_SUPPLIED',
                      'NOT_ESTABLISHED_IN_CONVERSATION')
# Words that appear in instructions or are generic English glue: excluded from the
# proper-noun probe so that ordinary prose is not mistaken for a fabricated entity.
PROPER_STOPWORDS = {'This', 'That', 'These', 'Those', 'There', 'Then', 'They', 'Their',
                    'What', 'When', 'Where', 'Which', 'While', 'With', 'Without',
                    'Reply', 'JSON', 'None', 'Null', 'True', 'False', 'Text', 'Turn',
                    'The', 'Conversation', 'Speaker', 'Evidence', 'Answer', 'Summary',
                    'Claim', 'Relation', 'Unclear', 'Ambiguous', 'Exophoric', 'Same',
                    'Different', 'Agreement', 'Disagreement', 'Insufficient'}


def _ctx_tokens(context):
    return {'labels': set(LABEL_RE.findall(context)),
            'numbers': set(NUMBER_RE.findall(context)),
            'placeholders': set(PLACEHOLDER_RE.findall(context)),
            'speakers': set(SPEAKER_RE.findall(context)),
            'has_url': bool(URL_RE.search(context)),
            'propers': set(PROPER_RE.findall(context)),
            'lower': context.lower()}


def triage(case, res):
    'Separate grounding failures from retrieval/format failures. Candidates only.'
    ctx = _ctx_tokens(case['input_context'])
    text = res.get('visible_output_after_thinking_strip') or ''
    parsed = res.get('parsed_output')
    gold = (case.get('reviewed_final_value') or {}).get('expected_answer')
    if gold is None:
        gold = (case.get('original_provisional_value') or {}).get('provisional_expected_answer')
    fam = case['task_family']

    hallu, wrong = [], []

    # ---- GROUNDING FAILURES (hallucination candidates) -------------------
    for lab in set(LABEL_RE.findall(text)):
        if lab not in ctx['labels']:
            hallu.append('FABRICATED_TURN_LABEL:%s' % lab)
    for num in set(NUMBER_RE.findall(text)):
        if num not in ctx['numbers'] and len(num) > 1:
            hallu.append('NUMBER_NOT_IN_CONTEXT:%s' % num)
    for ph in set(PLACEHOLDER_RE.findall(text)):
        if ph not in ctx['placeholders']:
            hallu.append('FABRICATED_PLACEHOLDER:%s' % ph)
    for sp in set(SPEAKER_RE.findall(text)):
        if sp not in ctx['speakers']:
            hallu.append('FABRICATED_SPEAKER_ID:%s' % sp)
    if URL_RE.search(text) and not ctx['has_url']:
        hallu.append('URL_NOT_IN_CONTEXT')
    for pr in set(PROPER_RE.findall(text)):
        if pr not in ctx['propers'] and pr not in PROPER_STOPWORDS:
            hallu.append('ENTITY_NOT_IN_CONTEXT:%s' % pr)

    abstained = any(m in text.upper() for m in ABSTENTION_MARKERS)

    # Answering a question the evidence cannot support is the canonical
    # answerability hallucination: the content is necessarily invented.
    if case.get('answerability_label') == 'INSUFFICIENT' and not abstained and text.strip():
        hallu.append('ANSWERED_WHEN_EVIDENCE_INSUFFICIENT')
    if fam == 'source_overreach_probe' and not abstained and text.strip():
        hallu.append('SOURCE_OVERREACH_DESCRIBED_UNSUPPLIED_LINK')
    if fam == 'missing_context_abstention' and not abstained and text.strip():
        hallu.append('RECONSTRUCTED_MISSING_CONTENT')
    if fam == 'placeholder_continuity' and re.search(r'\b(he|she|his|her|man|woman|wife|husband)\b',
                                                     text, re.IGNORECASE):
        hallu.append('PRIVACY_ATTRIBUTED_IDENTITY_TO_PLACEHOLDER')

    # ---- RETRIEVAL / FORMAT FAILURES (NOT hallucinations) ----------------
    if res.get('runtime_error'):
        wrong.append('RUNTIME_ERROR')
    elif parsed is None:
        wrong.append('FORMAT_ERROR_UNPARSEABLE_JSON')
    elif not text.strip():
        wrong.append('EMPTY_OUTPUT')
    if isinstance(parsed, dict) and case.get('expected_output_format'):
        missing = [k for k in case['expected_output_format'] if k not in parsed]
        if missing:
            wrong.append('SCHEMA_MISSING_KEYS:%s' % ','.join(missing))
    if res.get('truncated'):
        wrong.append('TRUNCATED_AT_TOKEN_BUDGET')

    matches_gold = None
    if isinstance(parsed, dict) and isinstance(gold, dict):
        if 'labels' in gold:
            pred = parsed.get('labels') if isinstance(parsed.get('labels'), list) else []
            pred_s, gold_s = set(map(str, pred)), set(map(str, gold['labels']))
            matches_gold = pred_s == gold_s
            if not matches_gold:
                # every predicted label exists in the thread -> retrieval, not invention
                if pred_s and pred_s < gold_s:
                    wrong.append('OMISSION_MISSING_%d_LABELS' % len(gold_s - pred_s))
                elif pred_s - gold_s and (pred_s - gold_s) <= ctx['labels']:
                    wrong.append('WRONG_EXISTING_ITEM_EXTRA_LABELS')
        elif 'label' in gold:
            pred = str(parsed.get('label', ''))
            matches_gold = pred == str(gold['label'])
            if not matches_gold and pred in ctx['labels']:
                wrong.append('WRONG_EXISTING_ITEM_LABEL')
        elif 'answer' in gold:
            matches_gold = (str(parsed.get('answer', '')).strip().lower()
                            == str(gold['answer']).strip().lower())

    hallu = sorted(set(hallu))
    wrong = sorted(set(wrong))

    if hallu:
        triage_class = 'HALLUCINATION_CANDIDATE'
    elif matches_gold is True:
        triage_class = 'MATCHES_PROVISIONAL_GOLD'
    elif wrong:
        triage_class = 'NON_HALLUCINATION_ERROR_CANDIDATE'
    elif case.get('gold_type') != 'MACHINE_DERIVED_VERIFIABLE':
        triage_class = 'REQUIRES_HUMAN_LABEL'
    else:
        triage_class = 'REQUIRES_HUMAN_LABEL'

    return {
        'run_id': res['run_id'], 'case_id': case['case_id'], 'thread_id': case['thread_id'],
        'post_id': case.get('post_id'), 'stratum': case.get('stratum'),
        'model_id': res['model_id'], 'model_key': res['model_key'],
        'task_family': fam, 'task_class': case['task_class'],
        'task_variant': case.get('task_variant'),
        'answerability_label': case.get('answerability_label'),
        'gold_type': case.get('gold_type'),
        'triage_class': triage_class,
        'hallucination_signals': ';'.join(hallu),
        'hallucination_signal_count': len(hallu),
        'non_hallucination_signals': ';'.join(wrong),
        'non_hallucination_signal_count': len(wrong),
        'abstained': abstained,
        'abstention_expected': case.get('answerability_label') == 'INSUFFICIENT',
        'matches_provisional_gold': matches_gold,
        'json_valid': parsed is not None, 'parse_method': res.get('parse_method'),
        'output_token_count': res.get('output_token_count'),
        'contained_thinking_block': res.get('contained_thinking_block'),
        'mean_token_logprob': res.get('mean_token_logprob'),
        'mean_vocab_entropy': res.get('mean_vocab_entropy'),
        # input characteristics carried through so Section 8 can cross-tabulate them
        **{'auto__' + k: v for k, v in (case.get('input_characteristics') or {}).items()},
        'human_label_required': True,
        'note': 'CANDIDATE ONLY. String absence is evidence, not proof, of ungroundedness.',
    }


def load_all_results():
    rows = []
    if RESULTS_PATH.exists():
        with open(RESULTS_PATH, encoding='utf-8') as fh:
            for line in fh:
                try:
                    rows.append(json.loads(line))
                except Exception:
                    continue
    return rows


ALL_RESULTS = load_all_results()
TRIAGE_ROWS = [triage(CASE_BY_ID[r['case_id']], r)
               for r in ALL_RESULTS if r.get('case_id') in CASE_BY_ID]
write_csv(out('gpu_hallucination_triage.csv'), TRIAGE_ROWS)

print('triaged responses:', len(TRIAGE_ROWS), 'from', len(ALL_RESULTS), 'stored results')
print('\ntriage classes:')
for k, v in Counter(r['triage_class'] for r in TRIAGE_ROWS).most_common():
    print('   %-38s %5d' % (k, v))

_sig = Counter()
for r in TRIAGE_ROWS:
    for s in (r['hallucination_signals'].split(';') if r['hallucination_signals'] else []):
        _sig[s.split(':')[0]] += 1
print('\nhallucination signal types (candidates):')
for k, v in _sig.most_common():
    print('   %-46s %5d' % (k, v))

_wsig = Counter()
for r in TRIAGE_ROWS:
    for s in (r['non_hallucination_signals'].split(';') if r['non_hallucination_signals'] else []):
        _wsig[s.split(':')[0]] += 1
print('\nnon-hallucination error types (wrong answers, NOT hallucinations):')
for k, v in _wsig.most_common():
    print('   %-46s %5d' % (k, v))

triaged responses: 0 from 0 stored results

triage classes:

hallucination signal types (candidates):

non-hallucination error types (wrong answers, NOT hallucinations):


In [18]:
# --- Section 8: where does each model hallucinate? ------------------------
# These tables exist to SUGGEST candidate benchmark dimensions. They are computed
# from automatic candidates, not human labels, so nothing here is a finding.

def rate(rows, pred):
    n = len(rows)
    return (sum(1 for r in rows if pred(r)) / n) if n else None


def _is_h(r):
    return r['triage_class'] == 'HALLUCINATION_CANDIDATE'


def _is_w(r):
    return r['triage_class'] == 'NON_HALLUCINATION_ERROR_CANDIDATE'


BREAKDOWN_ROWS = []


def breakdown(facet_name, keyfn):
    groups = defaultdict(list)
    for r in TRIAGE_ROWS:
        groups[str(keyfn(r))].append(r)
    for value, rows in sorted(groups.items()):
        BREAKDOWN_ROWS.append({
            'facet': facet_name, 'value': value, 'responses': len(rows),
            'unique_cases': len({r['case_id'] for r in rows}),
            'unique_threads': len({r['thread_id'] for r in rows}),
            'unique_posts': len({r['post_id'] for r in rows}),
            'hallucination_candidates': sum(1 for r in rows if _is_h(r)),
            'hallucination_candidate_rate': round(rate(rows, _is_h), 4),
            'non_hallucination_errors': sum(1 for r in rows if _is_w(r)),
            'non_hallucination_error_rate': round(rate(rows, _is_w), 4),
            'matches_gold': sum(1 for r in rows
                                if r['triage_class'] == 'MATCHES_PROVISIONAL_GOLD'),
            'json_valid_rate': round(rate(rows, lambda r: r['json_valid']), 4),
            'abstention_when_expected': sum(1 for r in rows
                                            if r['abstention_expected'] and r['abstained']),
            'abstention_opportunities': sum(1 for r in rows if r['abstention_expected']),
            'mean_entropy': round(
                sum(r['mean_vocab_entropy'] for r in rows
                    if r['mean_vocab_entropy'] is not None)
                / max(sum(1 for r in rows if r['mean_vocab_entropy'] is not None), 1), 4),
            'basis': 'AUTOMATIC_CANDIDATES_NOT_HUMAN_LABELS',
        })


if TRIAGE_ROWS:
    breakdown('model', lambda r: r['model_id'])
    breakdown('task_family', lambda r: r['task_family'])
    breakdown('task_class', lambda r: r['task_class'])
    breakdown('answerability', lambda r: r['answerability_label'])
    breakdown('stratum', lambda r: r['stratum'])
    for _c in ('auto__cand_language_profile', 'auto__text_length_bin',
               'auto__single_or_multi_node', 'auto__branching',
               'auto__has_url', 'auto__has_emoji', 'auto__placeholder_recurrence',
               'auto__empty_root_with_descendants', 'auto__informal_spelling_signal',
               'auto__cand_km_fr_switch', 'auto__cand_km_en_switch',
               'auto__fragmentary_text', 'auto__cand_referring_expression_present'):
        if any(_c in r for r in TRIAGE_ROWS):
            breakdown(_c.replace('auto__', 'input:'), lambda r, c=_c: r.get(c))
    write_csv(out('gpu_hallucination_breakdown.csv'), BREAKDOWN_ROWS)

    print('MODEL COMPARISON (hallucination candidates vs ordinary wrong answers)')
    print('-' * 88)
    print('%-34s %5s %10s %10s %9s' % ('model', 'n', 'hallu', 'wrong', 'json_ok'))
    for r in [b for b in BREAKDOWN_ROWS if b['facet'] == 'model']:
        print('%-34s %5d %9.1f%% %9.1f%% %8.1f%%'
              % (r['value'][:34], r['responses'],
                 100 * r['hallucination_candidate_rate'],
                 100 * r['non_hallucination_error_rate'],
                 100 * r['json_valid_rate']))

    print('\nTASK FAMILY (where hallucination candidates concentrate)')
    print('-' * 88)
    print('%-34s %5s %10s %10s' % ('task family', 'n', 'hallu', 'wrong'))
    _fam = [b for b in BREAKDOWN_ROWS if b['facet'] == 'task_family']
    for r in sorted(_fam, key=lambda x: -x['hallucination_candidate_rate']):
        print('%-34s %5d %9.1f%% %9.1f%%'
              % (r['value'][:34], r['responses'],
                 100 * r['hallucination_candidate_rate'],
                 100 * r['non_hallucination_error_rate']))

    print('\nINPUT CONDITIONS (candidate benchmark dimensions, ranked by signal)')
    print('-' * 88)
    print('%-40s %-12s %5s %10s' % ('input condition', 'value', 'n', 'hallu'))
    _inp = [b for b in BREAKDOWN_ROWS if b['facet'].startswith('input:')
            and b['responses'] >= 10]
    for r in sorted(_inp, key=lambda x: -x['hallucination_candidate_rate'])[:25]:
        print('%-40s %-12s %5d %9.1f%%'
              % (r['facet'][:40], str(r['value'])[:12], r['responses'],
                 100 * r['hallucination_candidate_rate']))

    print('\nABSTENTION BEHAVIOUR (cases where the correct answer is to refuse)')
    print('-' * 88)
    for r in [b for b in BREAKDOWN_ROWS if b['facet'] == 'model']:
        _opp = r['abstention_opportunities']
        _got = r['abstention_when_expected']
        print('%-34s abstained %d / %d (%.1f%%)'
              % (r['value'][:34], _got, _opp, 100.0 * _got / _opp if _opp else 0.0))
    print('\n[INTERPRETATION] Failing to abstain is a grounding failure: the model must')
    print('                 invent the content it supplies. Choosing the wrong existing')
    print('                 turn is not. Keep them apart when designing dimensions.')
else:
    print('No triaged responses yet. Run Section 6 first.')

No triaged responses yet. Run Section 6 first.


In [19]:
# --- Section 9: blinded annotation export ---------------------------------
# Model identity is hidden; presentation order is a deterministic per-case hash.
# The key is written to restricted/ and must not travel with the workbook.
HALLUCINATION_LABELS = [
    'UNSUPPORTED_ADDITION', 'CONTRADICTS_EVIDENCE', 'FABRICATED_ENTITY',
    'FABRICATED_ATTRIBUTE_RELATION', 'SOURCE_OVERREACH', 'UNJUSTIFIED_RESOLUTION',
    'ANSWERABILITY_FAILURE', 'PRIVACY_FAILURE',
]
NON_HALLUCINATION_LABELS = [
    'SUPPORTED_CORRECT', 'WRONG_EXISTING_ITEM', 'OMISSION', 'MISINTERPRETATION',
    'LANGUAGE_OR_ORTHOGRAPHY_DISTORTION', 'STRUCTURE_ERROR', 'FORMAT_ERROR',
    'REFUSAL_OR_NONANSWER', 'AMBIGUOUS_OR_UNGRADABLE', 'NEW_FAILURE_PATTERN',
]
ALL_LABELS = HALLUCINATION_LABELS + NON_HALLUCINATION_LABELS

BLINDING_SALT = seeded_rank('blinding-salt', RUN_ID)[:32]
BLINDED_ID = {m: 'SYS-%s' % sha256_text(BLINDING_SALT + '|' + m)[:6].upper()
              for m in sorted({r['model_id'] for r in ALL_RESULTS})}
_TRIAGE_BY = {(r['case_id'], r['model_id']): r for r in TRIAGE_ROWS}

ANNOTATION_ROWS = []
for res in ALL_RESULTS:
    case = CASE_BY_ID.get(res.get('case_id'))
    if case is None:
        continue
    t = _TRIAGE_BY.get((res['case_id'], res['model_id']), {})
    ANNOTATION_ROWS.append({
        'blinded_system_id': BLINDED_ID.get(res['model_id'], 'SYS-UNKNOWN'),
        'presentation_order_key': seeded_rank('blind', res['case_id'], res['model_id'])[:12],
        'case_id': case['case_id'], 'thread_id': case['thread_id'],
        'task_class': case['task_class'], 'task': case['task_family'],
        'task_variant': case.get('task_variant'), 'gold_type': case.get('gold_type'),
        'supplied_context': case['input_context'][:4000],
        'question': case['instruction'],
        'response': (res.get('visible_output_after_thinking_strip') or '')[:4000],
        'answerability_provisional': case.get('answerability_label'),
        'evidence_expected_provisional': ';'.join(case.get('evidence_order') or []),
        # machine triage shown as a PROMPT for the annotator, never as an answer
        'machine_triage_class': t.get('triage_class', ''),
        'machine_hallucination_signals': t.get('hallucination_signals', ''),
        'machine_other_error_signals': t.get('non_hallucination_signals', ''),
        # --- annotator fields (blank by design) ---
        'is_hallucination_yes_no': '',
        'hallucination_label': '',
        'non_hallucination_label': '',
        'secondary_labels': '',
        'unsupported_span_char_start': '', 'unsupported_span_char_end': '',
        'unsupported_span_text': '',
        'why_unsupported': '',
        'severity_minor_material_critical': '',
        'ambiguity_flag': '', 'ungradable_flag': '', 'cultural_dependence_flag': '',
        'candidate_dimension_suggested': '',
        'annotator_id': '', 'is_second_annotation': '',
        'annotator_confidence_1_5': '', 'notes': '',
        'adjudication_status': 'PENDING',
    })
ANNOTATION_ROWS.sort(key=lambda r: (r['case_id'], r['presentation_order_key']))

INSTRUCTIONS = [
    {'step': 1, 'instruction': 'You are judging responses, not models. Identity is hidden.'},
    {'step': 2, 'instruction': 'Judge each response ONLY against the supplied context.'},
    {'step': 3, 'instruction': 'HALLUCINATION = asserted content absent from, or contradicting, the context.'},
    {'step': 4, 'instruction': 'WRONG ANSWER = the content exists in the context but the choice is wrong.'},
    {'step': 5, 'instruction': 'Picking the wrong existing turn is NOT a hallucination. Inventing a turn IS.'},
    {'step': 6, 'instruction': 'Omission, truncation and bad JSON are NOT hallucinations.'},
    {'step': 7, 'instruction': 'Answering when the evidence is insufficient IS a hallucination.'},
    {'step': 8, 'instruction': 'Describing a linked page that was never supplied IS a hallucination.'},
    {'step': 9, 'instruction': 'For diagnostic tasks AMBIGUOUS / EXOPHORIC / UNCLEAR can be the correct answer.'},
    {'step': 10, 'instruction': 'Give char_start/char_end for every unsupported span you mark.'},
    {'step': 11, 'instruction': 'machine_* columns are hints from a string check. They are often wrong. Overrule them.'},
    {'step': 12, 'instruction': 'Never infer the real identity behind a PERSON_ placeholder or speaker id.'},
    {'step': 13, 'instruction': 'Use candidate_dimension_suggested to name the input condition you think caused it.'},
]

VOCAB_ROWS = ([{'label': l, 'is_hallucination': True,
                'definition': 'grounding failure: content not recoverable from the turns'}
               for l in HALLUCINATION_LABELS]
              + [{'label': l, 'is_hallucination': False,
                  'definition': 'not a grounding failure: retrieval, coverage or form'}
                 for l in NON_HALLUCINATION_LABELS])

_stem = out('gpu_human_annotation_blinded')
if HAVE_PANDAS:
    try:
        with pd.ExcelWriter(str(_stem) + '.xlsx', engine='openpyxl') as xw:
            pd.DataFrame(ANNOTATION_ROWS).to_excel(xw, sheet_name='annotation', index=False)
            pd.DataFrame(INSTRUCTIONS).to_excel(xw, sheet_name='instructions', index=False)
            pd.DataFrame(VOCAB_ROWS).to_excel(xw, sheet_name='label_vocabulary', index=False)
        _written = str(_stem) + '.xlsx'
    except Exception as exc:
        print('xlsx write failed (%s); falling back to CSV' % exc)
        _written = str(write_csv(str(_stem) + '.csv', ANNOTATION_ROWS))
else:
    _written = str(write_csv(str(_stem) + '.csv', ANNOTATION_ROWS))

write_csv(RESTRICTED_DIR / 'gpu_blinding_key_restricted.csv',
          [{'blinded_system_id': b, 'model_id': m, 'run_id': RUN_ID,
            'restricted': 'DO NOT DISTRIBUTE WITH THE ANNOTATION WORKBOOK'}
           for m, b in sorted(BLINDED_ID.items())])

RUN_SUMMARY = {
    'notebook': NOTEBOOK_NAME, 'version': NOTEBOOK_VERSION, 'run_id': RUN_ID,
    'started_utc': RUN_STARTED_UTC,
    'finished_utc': _dt.datetime.now(_dt.timezone.utc).isoformat(),
    'case_manifest_sha256': MANIFEST_SHA256, 'dataset_sha256': DATASET_SHA256,
    'prompt_version': PROMPT_VERSION, 'load_strategy': LOAD_STRATEGY,
    'decoding': DECODING_CONFIG, 'gpu': PREFLIGHT['gpu'],
    'models_requested': [m['model_id'] for m in MODELS],
    'models_run': sorted({r['model_id'] for r in ALL_RESULTS}),
    'cases_planned': len(CASES), 'responses_stored': len(ALL_RESULTS),
    'generations_this_session': len(INFERENCE_RESULTS),
    'load_errors': LOAD_ERRORS, 'run_errors': RUN_ERRORS[:100],
    'triage_classes': dict(Counter(r['triage_class'] for r in TRIAGE_ROWS)),
    'verification': VERIFICATION_ROWS,
    'limits': [
        'Triage emits CANDIDATES from string matching. It cannot detect a paraphrased '
        'invention, and it flags legitimate paraphrase as if it were unsupported.',
        'No hallucination rate here is a finding. Human labels are required.',
        'Blinding is imperfect: style, verbosity and thinking traces can identify a system.',
        'Greedy decoding only; a single sample per case says nothing about variance.',
        'Precision is an experimental variable: do not pool quantized and bf16 runs.',
    ],
}
write_json(out('gpu_run_summary.json'), RUN_SUMMARY)

print('annotation workbook :', _written)
print('rows to annotate    :', len(ANNOTATION_ROWS))
print('blinded systems     :', len(BLINDED_ID))
print('blinding key        :', RESTRICTED_DIR / 'gpu_blinding_key_restricted.csv')
print('\nartefacts written to', OUTPUT_DIR)
for _n in ('pilot_inference_results.jsonl', 'pilot_token_uncertainty.jsonl',
           'gpu_model_verification.csv', 'gpu_run_completeness.csv',
           'gpu_hallucination_triage.csv', 'gpu_hallucination_breakdown.csv',
           'gpu_run_summary.json'):
    _p = out(_n)
    print('   %-42s %s' % (_n, ('%d bytes' % _p.stat().st_size) if _p.exists() else 'MISSING'))

annotation workbook : /workspace/mauritian_creole_nlp/gpu_human_annotation_blinded.xlsx
rows to annotate    : 0
blinded systems     : 0
blinding key        : /workspace/mauritian_creole_nlp/restricted/gpu_blinding_key_restricted.csv

artefacts written to /workspace/mauritian_creole_nlp
   pilot_inference_results.jsonl              0 bytes
   pilot_token_uncertainty.jsonl              0 bytes
   gpu_model_verification.csv                 1557 bytes
   gpu_run_completeness.csv                   2005 bytes
   gpu_hallucination_triage.csv               7 bytes
   gpu_hallucination_breakdown.csv            MISSING
   gpu_run_summary.json                       5705 bytes


## How to read the output

### The two files that matter

* **`gpu_hallucination_triage.csv`** — one row per response, with `hallucination_signals`
  and `non_hallucination_signals` in **separate columns**. Sort by `triage_class` to find
  the grounding failures.
* **`gpu_hallucination_breakdown.csv`** — hallucination-candidate rate by model, task
  family and **input condition**. The `input:*` rows are your raw material for benchmark
  dimensions: an input condition with a high candidate rate and enough threads and posts
  behind it is worth proposing.

### Reading the signals

| Signal | Why it is a grounding failure |
|---|---|
| `FABRICATED_TURN_LABEL` | cites a turn that does not exist in the thread |
| `FABRICATED_SPEAKER_ID` / `FABRICATED_PLACEHOLDER` | invents an identifier |
| `ENTITY_NOT_IN_CONTEXT` | names a person, place or organisation never mentioned |
| `NUMBER_NOT_IN_CONTEXT` | asserts a quantity or date with no source |
| `ANSWERED_WHEN_EVIDENCE_INSUFFICIENT` | the answer is necessarily invented |
| `SOURCE_OVERREACH_...` | describes a linked page that was never supplied |
| `RECONSTRUCTED_MISSING_CONTENT` | rebuilds an empty root turn |
| `PRIVACY_ATTRIBUTED_IDENTITY_TO_PLACEHOLDER` | assigns gender or biography to `PERSON_nnnnnn` |

And the contrast set, which must **never** be counted as hallucination:
`WRONG_EXISTING_ITEM_*` (retrieval), `OMISSION_*` (coverage),
`FORMAT_ERROR_*` / `SCHEMA_MISSING_KEYS` / `TRUNCATED_*` (form).

### What the triage cannot do

It is a **string check**. It will:

* **miss** a fluent invention that reuses only words already in the context — the most
  dangerous hallucination class, and the one your annotators exist to catch;
* **over-flag** ordinary paraphrase, translation and any capitalised word it does not
  recognise.

Treat `machine_triage_class` as a sorting aid for the annotation queue, nothing more. The
`hallucination_candidate_rate` is **not** a hallucination rate.

### Next step

1. Annotate `gpu_human_annotation_blinded.xlsx`, filling `is_hallucination_yes_no`, the
   label columns and the unsupported spans.
2. Rename it `pilot_human_annotation_completed.xlsx` and drop it in `km_pilot_outputs/`.
3. Re-run Sections 23–24 of the main notebook, which will cross-tabulate the **human**
   labels against input characteristics and propose candidate dimensions properly.